# Exploratory Data Analysis (EDA)

## Objetivo

En los sistemas Retrieval-Augmented Generation (RAG), la fase de Exploratory Data Analysis (EDA) tiene como objetivo comprender la estructura, calidad y contenido de los documentos que servirán como base de conocimiento del chatbot.

A diferencia de los proyectos tradicionales de Machine Learning, donde el EDA se centra en analizar variables y distribuciones para construir modelos predictivos, en un sistema RAG el análisis se enfoca en:

- Comprender el tipo de documentos disponibles
- Evaluar su calidad textual
- Identificar ruido o información irrelevante
- Analizar la longitud y estructura del contenido
- Determinar la mejor estrategia de chunking para embeddings
- Detectar posibles problemas de extracción de texto

## Objetivos específicos

Durante esta fase se buscará:

1. Analizar la cantidad y tipos de documentos disponibles.
2. Evaluar el tamaño y longitud del contenido textual.
3. Detectar posibles errores de extracción (texto vacío, caracteres extraños).
4. Analizar la distribución de longitud de documentos.
5. Comprender la temática general del corpus.

Este análisis permitirá preparar correctamente los datos antes de la etapa de **preprocesamiento y generación de embeddings**.

In [2]:
# =============================================
# Configuración inicial del entorno — EDA RAG
# =============================================

# Librerías principales
import os
from pathlib import Path
import random
import pandas as pd
import numpy as np

# Visualización
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from IPython.display import display, Markdown

# Extracción y limpieza de PDFs
from pypdf import PdfReader
import pdfplumber
import re
import unicodedata

# Procesamiento de texto
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')

# Utilidades del sistema
import warnings
warnings.filterwarnings('ignore')

# Configuración gráfica
sns.set(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\lugod\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [ ]:
# ==============================
# 1. Configuración y carga de archivos
# ==============================

RAW_DIR = "../data/raw"
files = os.listdir(RAW_DIR)

print("Total de documentos:", len(files))

# ==============================
# 2. Extracción de información básica de cada archivo
# ==============================

data = []
for file in files:
    path = os.path.join(RAW_DIR, file)
    size_mb = os.path.getsize(path) / (1024 * 1024)
    
    data.append({
        "document": file,
        "size_mb": round(size_mb, 2)
    })

df_files = pd.DataFrame(data)

# ==============================
# 3. Inspección rápida de los datos
# ==============================

print(df_files)
print("\nDescripción estadística de tamaños:")
print(df_files.describe())

Total de documentos: 15
                                         document  size_mb
0              bond_fund_risk_monetary_policy.pdf     2.92
1                                eurosistemas.pdf     1.61
2               eurosystem_monetary_framework.pdf     0.32
3    evolucion_economica_financiera_monetaria.pdf     1.21
4                          glosario_economico.pdf     0.13
5        implications_for_financial_stability.pdf     1.37
6                        macroeconomia_ingles.pdf     0.85
7   money_demand_non_financial_corporationpdf.pdf     4.24
8           reporte_anual_resumen_2023_ingles.pdf     0.43
9           reporte_anual_resumen_2024_ingles.pdf     0.46
10                respuesta_politica_pandemia.pdf     0.63
11               stabalecoins_monetary_policy.pdf     3.41
12                        the_art_of_taxation.pdf     0.09
13                      the_labor_market_euro.pdf     3.32
14                    understanding_inflation.pdf     2.31

Descripción estadística de tama

In [6]:
# ==============================
# Análisis de contenido: longitud del texto extraído
# ==============================

# Carpeta con los PDFs
RAW_DIR = "../data/raw"

# Lista para guardar resultados
eda_results = []

# Recorrer todos los PDFs
for filename in os.listdir(RAW_DIR):
    if filename.lower().endswith(".pdf"):
        path = os.path.join(RAW_DIR, filename)
        result = {"document": filename, "status": "", "text_length": 0}
        
        try:
            reader = PdfReader(path)
            text = ""
            
            # Extraer texto de todas las páginas
            for page in reader.pages:
                page_text = page.extract_text()
                if page_text:
                    text += page_text
            
            # Guardar longitud del texto
            result["text_length"] = len(text.strip())
            
            # Clasificar PDF según cantidad de texto
            if len(text.strip()) > 50:
                result["status"] = "OK"
            else:
                result["status"] = "WARNING: poco texto"
        
        except Exception as e:
            result["status"] = f"ERROR: {e}"
        
        eda_results.append(result)

# Convertir a DataFrame
df_eda = pd.DataFrame(eda_results)

# Mostrar resultados
df_eda

,document,status,text_length
0,bond_fund_risk_monetary_policy.pdf,OK,112413
1,eurosistemas.pdf,OK,70319
2,eurosystem_monetary_framework.pdf,OK,81275
3,evolucion_economica_financiera_monetaria.pdf,OK,102446
4,glosario_economico.pdf,OK,6794
5,implications_for_financial_stability.pdf,OK,136762
6,macroeconomia_ingles.pdf,OK,89951
7,money_demand_non_financial_corporationpdf.pdf,OK,83132
8,reporte_anual_resumen_2023_ingles.pdf,OK,129882
9,reporte_anual_resumen_2024_ingles.pdf,OK,140277


In [8]:
# ==============================
# Análisis de contenido: detección de caracteres corruptos
# ==============================

# Lista para resultados de revisión de caracteres
char_check_results = []

for row in df_eda['document']:
    path = os.path.join(RAW_DIR, row)
    try:
        reader = PdfReader(path)
        text = ""
        for page in reader.pages:
            page_text = page.extract_text()
            if page_text:
                text += page_text

        sample_text = text[:1000]  # Revisar primeros 1000 caracteres
        # Buscar caracteres fuera del rango ASCII imprimible
        strange_chars = re.findall(r'[^\x20-\x7E\n\r\t]', sample_text)

        status = "OK" if not strange_chars else f"WARNING: {len(strange_chars)} caracteres extraños"
        
        char_check_results.append({
            "document": row,
            "char_check": status
        })
    
    except Exception as e:
        char_check_results.append({
            "document": row,
            "char_check": f"ERROR: {e}"
        })

df_char_check = pd.DataFrame(char_check_results)

# Mostrar solo el estado de caracteres extraños
df_char_check

,document,char_check
0,bond_fund_risk_monetary_policy.pdf,WARNING: 5 caracteres extraños
1,eurosistemas.pdf,WARNING: 16 caracteres extraños
2,eurosystem_monetary_framework.pdf,WARNING: 1 caracteres extraños
3,evolucion_economica_financiera_monetaria.pdf,WARNING: 27 caracteres extraños
4,glosario_economico.pdf,WARNING: 29 caracteres extraños
5,implications_for_financial_stability.pdf,OK
6,macroeconomia_ingles.pdf,WARNING: 2 caracteres extraños
7,money_demand_non_financial_corporationpdf.pdf,WARNING: 2 caracteres extraños
8,reporte_anual_resumen_2023_ingles.pdf,WARNING: 1 caracteres extraños
9,reporte_anual_resumen_2024_ingles.pdf,WARNING: 2 caracteres extraños


In [10]:
# ==============================
# Inspección manual de PDF con caracteres extraños
# ==============================

from pypdf import PdfReader
import os

pdf_file = "glosario_economico.pdf"
path = os.path.join(RAW_DIR, pdf_file)

reader = PdfReader(path)
text = ""

# Extraer texto de todas las páginas
for i, page in enumerate(reader.pages):
    page_text = page.extract_text()
    if page_text:
        text += page_text
        # Mostrar solo los primeros 300 caracteres de cada página para inspección rápida
        print(f"--- Página {i+1} ---")
        print(page_text[:300])
        print("\n")

--- Página 1 ---
Categorías  del  Glosario  1.  C
ONCEPTOS
 B
ÁSICOS
 
DE
 E
CONOMÍA
 ●  Costo  de  Oportunidad :  Concepto  económico  fundamental  que  representa  el  valor  de  la  mejor  alternativa  sacrificada  al  tomar  una  decisión  con  recursos  limitados.  ●  Demanda :  Comprende  el  concepto  fundame


--- Página 2 ---
●  Producto  Interno  Bruto  (PIB) :  El  Producto  Interno  Bruto  (PIB)  es  el  indicador  económico  más  importante  para  medir  la  riqueza  de  los  países.  Conoce  cómo  se  calcula,  sus  tipos  y  por  qué  Estados  Unidos  lidera  con  $26.95  billones.  ●  Tasa  de  Desempleo :  Artícu


--- Página 3 ---
●  Bono :  Guía  exhaustiva  sobre  bonos:  definición,  características,  tipos  (gubernamentales  y  corporativos),  funcionamiento  del  mercado,  sistema  de  calificación  crediticia,  estrategias  de  inversión,  riesgos  y  tendencias  2025.  Incluye  análisis  de  rentabilidades  actuales  y




### Interpretación

El documento `glosario_economico.pdf` presenta el mayor número de caracteres detectados como "extraños". 
Sin embargo, la inspección manual muestra que el texto es legible y coherente.

Las advertencias provienen principalmente de:
- Saltos de línea generados por el layout del PDF.
- Viñetas o símbolos (`●`).
- Caracteres acentuados o especiales.

Esto es común al extraer texto desde PDFs, ya que estos almacenan contenido como posiciones en la página y no como párrafos estructurados.

En esta fase de EDA no se aplican transformaciones; el objetivo es únicamente diagnosticar la calidad del texto. 
La normalización de espacios, saltos de línea y caracteres se abordará posteriormente en la fase de **preprocesamiento antes del chunking y generación de embeddings**. Mejorar luego...

In [11]:
# ==============================
# Análisis de estructura: párrafos y saltos de línea
# ==============================

structure_results = []

for doc in df_eda['document']:
    path = os.path.join(RAW_DIR, doc)
    reader = PdfReader(path)

    text = ""
    for page in reader.pages:
        page_text = page.extract_text()
        if page_text:
            text += page_text

    paragraphs = text.split("\n")
    num_paragraphs = len([p for p in paragraphs if p.strip() != ""])

    structure_results.append({
        "document": doc,
        "paragraph_count": num_paragraphs
    })

df_structure = pd.DataFrame(structure_results)

df_structure

,document,paragraph_count
0,bond_fund_risk_monetary_policy.pdf,1607
1,eurosistemas.pdf,1470
2,eurosystem_monetary_framework.pdf,1195
3,evolucion_economica_financiera_monetaria.pdf,1376
4,glosario_economico.pdf,21
5,implications_for_financial_stability.pdf,1635
6,macroeconomia_ingles.pdf,1271
7,money_demand_non_financial_corporationpdf.pdf,1283
8,reporte_anual_resumen_2023_ingles.pdf,15002
9,reporte_anual_resumen_2024_ingles.pdf,16073


La mayoría de los documentos presentan entre 700 y 1,600 párrafos, lo que sugiere que la extracción de texto preserva la estructura básica del contenido.
Dos reportes anuales presentan valores significativamente mayores (~15,000), probablemente debido a saltos de línea frecuentes o contenido tabular. Este comportamiento es común en PDFs y no representa un problema crítico, ya que el proceso posterior de chunking reorganizará el texto para su uso en el sistema RAG. mejorar...

In [12]:
# ==============================
# Detección de posibles headers repetidos
# ==============================

from collections import Counter

header_results = []

for doc in df_eda['document']:
    
    path = os.path.join(RAW_DIR, doc)
    reader = PdfReader(path)

    headers = []

    for page in reader.pages:
        text = page.extract_text()
        if text:
            first_line = text.split("\n")[0].strip()
            headers.append(first_line)

    counter = Counter(headers)

    repeated_headers = {k:v for k,v in counter.items() if v > 2}

    header_results.append({
        "document": doc,
        "num_pages": len(reader.pages),
        "repeated_headers": len(repeated_headers)
    })

df_headers = pd.DataFrame(header_results)

df_headers

,document,num_pages,repeated_headers
0,bond_fund_risk_monetary_policy.pdf,53,1
1,eurosistemas.pdf,36,0
2,eurosystem_monetary_framework.pdf,31,0
3,evolucion_economica_financiera_monetaria.pdf,44,1
4,glosario_economico.pdf,3,0
5,implications_for_financial_stability.pdf,51,1
6,macroeconomia_ingles.pdf,39,1
7,money_demand_non_financial_corporationpdf.pdf,47,1
8,reporte_anual_resumen_2023_ingles.pdf,25,0
9,reporte_anual_resumen_2024_ingles.pdf,27,0


El análisis de las primeras líneas de cada página muestra que algunos documentos contienen un único header repetido, lo cual es común en informes académicos o institucionales. Sin embargo, la cantidad de texto repetido es baja y no representa una fuente significativa de ruido para el sistema RAG. La mayoría de los documentos no presentan patrones claros de headers repetidos. Mejorar...

In [13]:
# ==============================
# Longitud del texto por documento
# ==============================

length_results = []

for doc in df_eda['document']:
    
    path = os.path.join(RAW_DIR, doc)
    reader = PdfReader(path)

    text = ""
    
    for page in reader.pages:
        page_text = page.extract_text()
        if page_text:
            text += page_text

    word_count = len(text.split())

    length_results.append({
        "document": doc,
        "word_count": word_count
    })

df_length = pd.DataFrame(length_results)

df_length

,document,word_count
0,bond_fund_risk_monetary_policy.pdf,16698
1,eurosistemas.pdf,9825
2,eurosystem_monetary_framework.pdf,12800
3,evolucion_economica_financiera_monetaria.pdf,16374
4,glosario_economico.pdf,848
5,implications_for_financial_stability.pdf,20927
6,macroeconomia_ingles.pdf,13922
7,money_demand_non_financial_corporationpdf.pdf,13308
8,reporte_anual_resumen_2023_ingles.pdf,15336
9,reporte_anual_resumen_2024_ingles.pdf,16446


La mayoría de los documentos presentan entre 10,000 y 18,000 palabras, lo que indica que se trata principalmente de informes o artículos de longitud media. Solo algunos documentos son significativamente más cortos. Esta distribución es adecuada para un sistema RAG, ya que permitirá generar un número equilibrado de chunks durante el proceso de segmentación del texto. Mejorar...